# Bibi Desk: the strategy, end to end

This notebook is the research writeup behind the desk. It walks the full chain from a Kronos
forecast to a sized, risk managed position, with the math written out so it can be checked
rather than taken on faith.

The order is the same as the code:

1. Forecast with Kronos
2. Turn the forecast into a cost aware signal
3. Size with fractional Kelly
4. Frame the trade in R units and manage the risk
5. Measure the whole thing honestly with a backtest


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

## 1. The forecast

Kronos is an autoregressive model over tokenized candlesticks. Given a context window of the
last $L$ bars $x_{t-L:t}$, it produces a distribution over the next $H$ bars rather than a single
point. We draw $N$ sample paths and read two numbers off them: the expected forward log return
and its dispersion.

$$
\mu_t = \mathbb{E}\left[\, r_{t \to t+H} \mid \text{Kronos}(x_{t-L:t}) \,\right]
\qquad
\sigma_t^2 = \operatorname{Var}\left[\, r_{t \to t+H} \mid \text{Kronos}(x_{t-L:t}) \,\right]
$$

where $r_{t \to t+H} = \log(P_{t+H} / P_t)$. In code this is the mean and std across the sampled
closes. Here we stand in a synthetic Kronos with a small, noisy edge so the rest of the
notebook runs without the model weights.

In [ ]:
def kronos_forecast(context_close, n_samples=30, horizon=24, true_edge=0.004, vol=0.02):
    """Stand in for KronosForecaster.forecast(). Returns sampled forward log returns.
    The real model conditions on the full OHLCV context. Here we just sample a path
    whose mean carries a faint edge, which is all we need to exercise the pipeline.
    """
    drift = true_edge + 0.0008 * np.tanh(np.diff(np.log(context_close[-10:])).mean() * 50)
    paths = rng.normal(drift, vol, size=n_samples)
    return paths

context = 30000 * np.cumprod(1 + rng.normal(0, 0.01, 512))
paths = kronos_forecast(context)
mu, sigma = paths.mean(), paths.std(ddof=1)
print(f'mu     = {mu:+.4%}  (expected forward return)')
print(f'sigma  = {sigma:.4%}  (dispersion of the forecast)')
print(f'dir    = {"LONG" if mu > 0 else "SHORT"}')

## 2. From forecast to signal

A forecast is not a trade. The edge has to clear the cost of taking it, and it has to clear a
confidence floor so we are not paying fees to trade noise. Let the round trip cost in return
terms be

$$ c(q) = \text{fee}_{bps} \cdot 10^{-4} \;+\; \text{slippage}_{bps}(q) \cdot 10^{-4} $$

The net edge and the fire rule are

$$ s_t = \mu_t - c(q), \qquad \text{fire} \iff s_t > \kappa \, \sigma_t $$

$\kappa$ is the confidence floor. A larger $\kappa$ trades less often but at higher conviction.
This is the single most important knob for staying alive: most of the forecast distribution is
noise, and $\kappa \sigma_t$ is the threshold that says how far into the tail the edge has to
reach before we act.

In [ ]:
def signal(mu, sigma, fee_bps=10, slippage_bps=5, kappa=0.6):
    cost = (fee_bps + slippage_bps) * 1e-4
    s = mu - cost
    fire = s > kappa * sigma
    return s, fire

s, fire = signal(mu, sigma)
print(f'net edge s = {s:+.4%}')
print(f'threshold  = {0.6 * sigma:.4%}  (kappa * sigma)')
print(f'fire       = {fire}')

## 3. Position sizing: fractional Kelly

Given an edge with mean $\mu$ and variance $\sigma^2$, the Kelly fraction that maximizes the
expected log growth of the bankroll is, in the continuous (Gaussian) approximation,

$$ f^\star = \frac{\mu - c}{\sigma^2} $$

Full Kelly is famously too aggressive: it is the most volatile allocation that still grows, and
a model that is even a little overconfident in $\mu$ will overbet badly. So we scale by
$\lambda \in (0, 1]$ (we run $\lambda = \tfrac{1}{2}$, half Kelly) and then clamp the notional so
a single trade can never risk more than a fixed slice of equity.

$$ \text{notional} = \operatorname{clamp}\!\left( \lambda f^\star \cdot \text{equity},\; 0,\; \rho \cdot \text{equity} \right) $$

Half Kelly keeps roughly three quarters of the growth rate of full Kelly at a quarter of the
variance drag, which is the trade most desks actually want.

In [ ]:
def kelly_fraction(mu, sigma, cost=0.0015, lam=0.5, cap=0.25):
    if sigma <= 1e-9:
        return 0.0
    f_star = (mu - cost) / (sigma ** 2)
    f = lam * f_star
    return float(np.clip(f, 0.0, cap))

equity = 50_000
f = kelly_fraction(mu, sigma)
notional = f * equity
print(f'f_star      = {(mu - 0.0015) / sigma**2:.3f}')
print(f'f (half)    = {f:.3f}  (clamped)')
print(f'notional    = ${notional:,.0f} on ${equity:,.0f} equity')

### Why half Kelly, in one chart

The expected log growth as a function of the bet fraction $f$ is a downward parabola,
$g(f) = f\mu - \tfrac{1}{2} f^2 \sigma^2$, peaking at $f^\star$. The curve is flat near the top
and steep past it, so the asymmetry pays you to underbet.

In [ ]:
import numpy as np
mu_d, sig_d = 0.01, 0.04
f_star = mu_d / sig_d**2
fs = np.linspace(0, 1.6 * f_star, 200)
g = fs * mu_d - 0.5 * fs**2 * sig_d**2
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7,3))
    plt.plot(fs, g, color='#D8A84B')
    plt.axvline(f_star, ls='--', color='#888', label='full Kelly f*')
    plt.axvline(0.5*f_star, ls=':', color='#444', label='half Kelly')
    plt.xlabel('bet fraction f'); plt.ylabel('expected log growth g(f)')
    plt.legend(); plt.tight_layout()
except Exception as e:
    print('matplotlib not installed, skipping plot:', e)

## 4. Risk in R units

PnL in dollars is noisy and scale dependent, so the desk frames every trade in R, where 1R is
the dollar amount risked if the stop is hit. With an ATR based stop, $1R = m \cdot \text{ATR}$
for a multiplier $m$. A trade that targets $k$R has reward to risk $k$. Expectancy per trade is

$$ \mathbb{E}[R] = p \cdot k - (1 - p) \cdot 1 = p(k + 1) - 1 $$

for win probability $p$ and target $k$R. Positive expectancy needs $p > \tfrac{1}{k+1}$. At
$k = 1.7$ that break even hit rate is about 37%, which is why a 58% hit rate with $1.7$R targets
is a comfortable edge rather than a coin flip.

In [ ]:
def expectancy_R(p, k):
    return p * k - (1 - p) * 1.0

for p in (0.37, 0.45, 0.55, 0.584):
    print(f'p={p:.3f}  k=1.7R  ->  E[R] = {expectancy_R(p, 1.7):+.3f}R')

be = 1 / (1.7 + 1)
print(f'\nbreak even hit rate at 1.7R: {be:.1%}')

## 5. A toy backtest

Putting it together on synthetic bars: forecast, gate on the signal, size with half Kelly,
resolve the trade as a win or loss in R drawn from the modeled hit rate, and net out fees. The
point is not the equity curve (the data is fake) but that the accounting lines up with the math
above: expectancy in R, profit factor, and a Sharpe that is computed net of cost.

In [ ]:
def toy_backtest(n=5000, p_win=0.584, k=1.7, fee_bps=10, slippage_bps=5):
    cost = (fee_bps + slippage_bps) * 1e-4
    R_unit = 0.01            # risk 1% of equity per trade at 1R
    eq = 1.0
    curve, rs = [eq], []
    for _ in range(n):
        mu = rng.normal(0.004, 0.002)
        sigma = abs(rng.normal(0.02, 0.004))
        s, fire = signal(mu, sigma, fee_bps, slippage_bps)
        if not fire:
            continue
        win = rng.random() < p_win
        r = k if win else -1.0
        r -= cost / R_unit            # fees expressed in R
        rs.append(r)
        eq *= (1 + r * R_unit)
        curve.append(eq)
    return np.array(curve), np.array(rs)

curve, rs = toy_backtest()
wins = rs[rs > 0].sum()
losses = -rs[rs < 0].sum()
print(f'trades        {len(rs):,}')
print(f'hit rate      {(rs > 0).mean():.1%}')
print(f'expectancy    {rs.mean():+.3f} R')
print(f'profit factor {wins / losses:.2f}')
print(f'final equity  {curve[-1]:.2f}x')

In [ ]:
# Sharpe and max drawdown on the per trade R series, net of fees.
def sharpe(rs, trades_per_year=2000):
    if rs.std(ddof=1) == 0: return 0.0
    return rs.mean() / rs.std(ddof=1) * np.sqrt(trades_per_year)

def max_drawdown(curve):
    peak = np.maximum.accumulate(curve)
    return float((curve / peak - 1).min())

print(f'Sharpe (ann.)  {sharpe(rs):.2f}')
print(f'max drawdown   {max_drawdown(curve):.1%}')

## Takeaways

The desk is not clever, and that is the point. The only source of edge is the Kronos forecast.
Everything downstream (the cost gate, half Kelly, R framing, the kill switch) exists to keep a
small, real edge from being given back to fees, overbetting, or a bad day. The math is plain on
purpose so it can be audited, and the live engine in `execution-engine/` runs exactly this logic
with a Rust order router in front of the venue.

See `docs/strategy.md` for the prose version and `backtest/` for the event driven backtester
that replaces the toy loop above with real bars.